# SpyCloud Threat Landscape Analysis**Purpose:** Provide organizational-level visibility into credential exposure trends, breach intelligence, risk posture, and network security correlation using SpyCloud data integrated with Microsoft Sentinel.This notebook generates executive-ready dashboards and KPIs for security leadership.---

## Section 1: Setup & ConfigurationImport required libraries and connect to the Microsoft Sentinel workspace.

In [ ]:
# Install dependencies if needed# %pip install msticpy[all] pandas matplotlib plotly networkx azure-identityimport warningswarnings.filterwarnings("ignore")import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport networkx as nxfrom datetime import datetime, timedeltafrom IPython.display import display, HTML, Markdownimport msticpy as mpmp.init_notebook(namespace=globals(), verbosity=0)

In [ ]:
# Connect to Microsoft Sentinel workspaceqry_prov = QueryProvider("MSSentinel")qry_prov.connect(    connection_str="loganalytics://code().tenant('YOUR_TENANT_ID')"    ".workspace('YOUR_WORKSPACE_ID')")# Analysis parametersLOOKBACK_DAYS = 90START_TIME = datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)END_TIME = datetime.utcnow()ORG_DOMAINS = ["yourcompany.com"]  # Add your organization's email domainsprint(f"Connected successfully.")print(f"Analysis period: {START_TIME.date()} to {END_TIME.date()} ({LOOKBACK_DAYS} days)")

## Section 2: Exposure LandscapeAnalyze the overall exposure posture across the organization, including trends, severity, and geographic distribution.

In [ ]:
# Total exposure statistics over timeexposure_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| summarize    TotalExposures = count(),    UniqueUsers = dcount(email_s),    UniqueBreaches = dcount(source_id_d),    UniqueDomains = dcount(domain_s),    PlaintextCount = countif(password_type_s == "plaintext"),    HashedCount = countif(password_type_s has_any ("hashed", "hash")),    CrackedCount = countif(password_type_s == "cracked")"""total_stats = qry_prov.exec_query(exposure_query)if total_stats is not None and not total_stats.empty:    stats = total_stats.iloc[0]    display(Markdown("### Organizational Exposure Overview"))    fig = go.Figure()    metrics = ['TotalExposures', 'UniqueUsers', 'UniqueBreaches', 'UniqueDomains']    values = [int(stats[m]) for m in metrics]    labels = ['Total Exposures', 'Unique Users', 'Unique Breaches', 'Unique Domains']    fig.add_trace(go.Indicator(        mode="number", value=values[0],        title={"text": labels[0]},        domain={'x': [0, 0.25], 'y': [0.5, 1]}    ))    fig.add_trace(go.Indicator(        mode="number", value=values[1],        title={"text": labels[1]},        domain={'x': [0.25, 0.5], 'y': [0.5, 1]}    ))    fig.add_trace(go.Indicator(        mode="number", value=values[2],        title={"text": labels[2]},        domain={'x': [0.5, 0.75], 'y': [0.5, 1]}    ))    fig.add_trace(go.Indicator(        mode="number", value=values[3],        title={"text": labels[3]},        domain={'x': [0.75, 1], 'y': [0.5, 1]}    ))    # Password type breakdown    fig.add_trace(go.Indicator(        mode="number", value=int(stats['PlaintextCount']),        title={"text": "Plaintext Passwords"},        number={"font": {"color": "red"}},        domain={'x': [0, 0.33], 'y': [0, 0.4]}    ))    fig.add_trace(go.Indicator(        mode="number", value=int(stats['HashedCount']),        title={"text": "Hashed Passwords"},        number={"font": {"color": "orange"}},        domain={'x': [0.33, 0.66], 'y': [0, 0.4]}    ))    fig.add_trace(go.Indicator(        mode="number", value=int(stats['CrackedCount']),        title={"text": "Cracked Passwords"},        number={"font": {"color": "darkred"}},        domain={'x': [0.66, 1], 'y': [0, 0.4]}    ))    fig.update_layout(height=400, title="SpyCloud Exposure Dashboard")    fig.show()

In [ ]:
# Exposure trend over time (weekly)trend_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| summarize    NewExposures = count(),    UniqueUsers = dcount(email_s),    PlaintextPct = round(100.0 * countif(password_type_s == "plaintext") / count(), 1),    AvgSeverity = round(avg(severity_d), 2)  by Week = startofweek(TimeGenerated)| order by Week asc"""trend_df = qry_prov.exec_query(trend_query)if trend_df is not None and not trend_df.empty:    fig = make_subplots(        rows=2, cols=2,        subplot_titles=("New Exposures per Week", "Unique Affected Users",                        "Plaintext Password %", "Average Severity"),        vertical_spacing=0.12    )    fig.add_trace(go.Bar(x=trend_df['Week'], y=trend_df['NewExposures'],                         name='New Exposures', marker_color='steelblue'), row=1, col=1)    fig.add_trace(go.Scatter(x=trend_df['Week'], y=trend_df['UniqueUsers'],                             name='Unique Users', mode='lines+markers',                             line=dict(color='orange')), row=1, col=2)    fig.add_trace(go.Scatter(x=trend_df['Week'], y=trend_df['PlaintextPct'],                             name='Plaintext %', mode='lines+markers',                             fill='tozeroy', line=dict(color='red')), row=2, col=1)    fig.add_trace(go.Scatter(x=trend_df['Week'], y=trend_df['AvgSeverity'],                             name='Avg Severity', mode='lines+markers',                             line=dict(color='purple')), row=2, col=2)    fig.update_layout(height=600, title_text="Exposure Trends Over Time", showlegend=False)    fig.show()

In [ ]:
# Severity trend analysisseverity_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| extend SeverityBucket = case(    severity_d >= 20, "Critical (20+)",    severity_d >= 10, "High (10-19)",    severity_d >= 5, "Medium (5-9)",    "Low (0-4)")| summarize Count = count() by SeverityBucket, Week = startofweek(TimeGenerated)| order by Week asc"""severity_df = qry_prov.exec_query(severity_query)if severity_df is not None and not severity_df.empty:    fig = px.area(        severity_df, x='Week', y='Count', color='SeverityBucket',        title="Severity Distribution Over Time",        color_discrete_map={            "Critical (20+)": "darkred",            "High (10-19)": "red",            "Medium (5-9)": "orange",            "Low (0-4)": "gold"        }    )    fig.update_layout(height=400)    fig.show()

In [ ]:
# Top affected domainsdomain_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| where domain_s != ""| summarize    ExposureCount = count(),    UniqueUsers = dcount(email_s),    PlaintextPct = round(100.0 * countif(password_type_s == "plaintext") / count(), 1),    AvgSeverity = round(avg(severity_d), 1)  by domain_s| order by ExposureCount desc| take 20"""domain_df = qry_prov.exec_query(domain_query)if domain_df is not None and not domain_df.empty:    display(Markdown("### Top Affected Domains"))    fig = px.bar(        domain_df, x='domain_s', y='ExposureCount',        color='AvgSeverity', color_continuous_scale='RdYlGn_r',        title="Top 20 Affected Domains by Exposure Count",        hover_data=['UniqueUsers', 'PlaintextPct']    )    fig.update_layout(xaxis_tickangle=-45, height=450)    fig.show()

In [ ]:
# Geographic distribution of infectionsgeo_query = f"""SpyCloudCompassData_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| where ip_address_s != ""| extend GeoInfo = geo_info_from_ip_address(ip_address_s)| extend    Country = tostring(GeoInfo.country),    State = tostring(GeoInfo.state),    City = tostring(GeoInfo.city),    Latitude = todouble(GeoInfo.latitude),    Longitude = todouble(GeoInfo.longitude)| where Country != ""| summarize    InfectionCount = count(),    UniqueDevices = dcount(infected_machine_id_s),    UniqueUsers = dcount(email_s)  by Country, State, City, Latitude, Longitude| order by InfectionCount desc"""geo_df = qry_prov.exec_query(geo_query)if geo_df is not None and not geo_df.empty:    display(Markdown("### Geographic Distribution of Infections"))    fig = px.scatter_geo(        geo_df,        lat='Latitude', lon='Longitude',        size='InfectionCount',        color='InfectionCount',        hover_name='City',        hover_data=['Country', 'State', 'UniqueDevices', 'UniqueUsers'],        title="Infection Locations Worldwide",        projection='natural earth',        color_continuous_scale='Reds'    )    fig.update_layout(height=500)    fig.show()    # Country summary    country_summary = geo_df.groupby('Country').agg(        TotalInfections=('InfectionCount', 'sum'),        UniqueDevices=('UniqueDevices', 'sum')    ).sort_values('TotalInfections', ascending=False).head(10).reset_index()    fig2 = px.bar(country_summary, x='Country', y='TotalInfections',                  title="Top 10 Countries by Infection Count",                  color='UniqueDevices', color_continuous_scale='Blues')    fig2.show()

## Section 3: Breach IntelligenceAnalyze the breach catalog for patterns, malware family trends, and source intelligence.

In [ ]:
# Breach catalog analysiscatalog_query = f"""SpyCloudBreachCatalog_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| project    source_id_d,    title_s,    description_s,    type_s,    num_records_d,    spycloud_publish_date_t,    acquisition_date_t,    confidence_d,    site_s,    sensitive_source_b| distinct *| order by spycloud_publish_date_t desc"""catalog_df = qry_prov.exec_query(catalog_query)if catalog_df is not None and not catalog_df.empty:    display(Markdown(f"### Breach Catalog: {len(catalog_df)} breaches in analysis period"))    # Type distribution    type_dist = catalog_df['type_s'].value_counts().reset_index()    type_dist.columns = ['Breach Type', 'Count']    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "pie"}, {"type": "bar"}]],                        subplot_titles=("Breach Type Distribution", "Top 15 Breaches by Record Count"))    fig.add_trace(go.Pie(labels=type_dist['Breach Type'], values=type_dist['Count'],                         hole=0.3), row=1, col=1)    top_breaches = catalog_df.nlargest(15, 'num_records_d')    fig.add_trace(go.Bar(        x=top_breaches['title_s'].str[:25],        y=top_breaches['num_records_d'],        marker_color='steelblue'    ), row=1, col=2)    fig.update_layout(height=400, title_text="Breach Catalog Analysis")    fig.show()    display(catalog_df[['title_s', 'type_s', 'num_records_d', 'spycloud_publish_date_t',                        'confidence_d', 'sensitive_source_b']].head(20))

In [ ]:
# Malware family trendsmalware_query = f"""SpyCloudCompassData_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| where malware_family_s != ""| summarize    InfectionCount = count(),    UniqueDevices = dcount(infected_machine_id_s),    UniqueUsers = dcount(email_s),    FirstSeen = min(TimeGenerated),    LastSeen = max(TimeGenerated)  by malware_family_s| order by InfectionCount desc| take 20"""malware_df = qry_prov.exec_query(malware_query)if malware_df is not None and not malware_df.empty:    display(Markdown("### Malware Family Trends"))    fig = px.bar(        malware_df, x='malware_family_s', y='InfectionCount',        color='UniqueDevices', color_continuous_scale='Reds',        title="Top 20 Malware Families",        hover_data=['UniqueUsers', 'FirstSeen', 'LastSeen']    )    fig.update_layout(xaxis_tickangle=-45, height=450)    fig.show()    # Malware trend over time    malware_trend_query = f"""    SpyCloudCompassData_CL    | where TimeGenerated >= ago({LOOKBACK_DAYS}d)    | where malware_family_s != ""    | summarize Count = count() by malware_family_s, Week = startofweek(TimeGenerated)    | top-nested 5 of malware_family_s by sum(Count)    | top-nested of Week by Count = sum(Count)    """    malware_trend = qry_prov.exec_query(malware_trend_query)    if malware_trend is not None and not malware_trend.empty:        fig2 = px.line(            malware_trend, x='Week', y='Count', color='malware_family_s',            title="Top 5 Malware Family Trends Over Time"        )        fig2.update_layout(height=400)        fig2.show()

In [ ]:
# New vs historical breachesnew_vs_hist_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| extend PublishDate = todatetime(spycloud_publish_date_t)| extend IsNew = iff(PublishDate >= ago(30d), "New (Last 30 days)", "Historical")| summarize    Count = count(),    UniqueUsers = dcount(email_s),    AvgSeverity = round(avg(severity_d), 1)  by IsNew, Week = startofweek(TimeGenerated)| order by Week asc"""new_hist_df = qry_prov.exec_query(new_vs_hist_query)if new_hist_df is not None and not new_hist_df.empty:    display(Markdown("### New vs Historical Breach Exposures"))    fig = px.bar(        new_hist_df, x='Week', y='Count', color='IsNew',        barmode='group',        title="New vs Historical Breach Exposures Over Time",        color_discrete_map={"New (Last 30 days)": "red", "Historical": "steelblue"}    )    fig.update_layout(height=400)    fig.show()

In [ ]:
# Sensitive source trackingsensitive_query = f"""SpyCloudBreachCatalog_CL| where sensitive_source_b == true| join kind=inner (    SpyCloudBreachWatchlist_CL    | where TimeGenerated >= ago({LOOKBACK_DAYS}d)    | summarize ExposureCount = count(), UniqueUsers = dcount(email_s) by source_id_d) on source_id_d| project    source_id_d,    title_s,    type_s,    ExposureCount,    UniqueUsers,    num_records_d,    spycloud_publish_date_t| order by ExposureCount desc"""sensitive_df = qry_prov.exec_query(sensitive_query)if sensitive_df is not None and not sensitive_df.empty:    display(Markdown("### Sensitive Source Breaches"))    display(Markdown("These breaches are from sensitive sources and require careful handling."))    display(sensitive_df)else:    print("No sensitive source breaches found in current period.")

## Section 4: Organizational Risk ScoreCalculate a composite organizational risk score based on multiple factors including exposure rates, remediation effectiveness, and behavioral analytics.

In [ ]:
# Gather risk metricsrisk_metrics_query = f"""let ActiveExposures = SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago(30d)| summarize ActiveCount = count(), ActiveUsers = dcount(email_s);let PlaintextPct = SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago(30d)| summarize PlaintextPct = round(100.0 * countif(password_type_s == "plaintext") / count(), 1);let RemediationRate = SpyCloud_ConditionalAccessLogs_CL| where TimeGenerated >= ago(30d)| summarize Remediated = dcountif(UserPrincipalName_s, Result_s == "Success"),            Total = dcount(UserPrincipalName_s)| extend RemediationPct = round(100.0 * Remediated / Total, 1);let ReinfectionRate = SpyCloudCompassData_CL| where TimeGenerated >= ago(90d)| summarize InfectionDates = make_set(startofday(infected_time_t)) by infected_machine_id_s| extend InfectionCount = array_length(InfectionDates)| where InfectionCount > 1| summarize ReinfectedDevices = count(), TotalDevices = dcount(infected_machine_id_s);let UEBACorrelation = BehaviorAnalytics| where TimeGenerated >= ago(30d)| where InvestigationPriority > 5| join kind=inner (    SpyCloudBreachWatchlist_CL    | where TimeGenerated >= ago(30d)    | distinct email_s) on $left.UserPrincipalName == $right.email_s| summarize HighPriorityCorrelated = count();ActiveExposures| extend p = 1| join kind=fullouter (PlaintextPct | extend p = 1) on p| join kind=fullouter (RemediationRate | extend p = 1) on p| join kind=fullouter (ReinfectionRate | extend p = 1) on p| join kind=fullouter (UEBACorrelation | extend p = 1) on p| project ActiveCount, ActiveUsers, PlaintextPct, RemediationPct,          ReinfectedDevices, TotalDevices, HighPriorityCorrelated"""risk_metrics = qry_prov.exec_query(risk_metrics_query)if risk_metrics is not None and not risk_metrics.empty:    m = risk_metrics.iloc[0]    display(Markdown("### Risk Metric Components"))    display(risk_metrics)else:    # Use fallback empty metrics    m = pd.Series({        'ActiveCount': 0, 'ActiveUsers': 0, 'PlaintextPct': 0,        'RemediationPct': 100, 'ReinfectedDevices': 0,        'TotalDevices': 1, 'HighPriorityCorrelated': 0    })    print("Could not fetch all risk metrics. Using defaults.")

In [ ]:
# Calculate composite risk scoredef calculate_org_risk_score(metrics):    """    Composite risk score (0-100):    - Active exposures weight: 25%    - Plaintext password %: 25%    - Remediation rate (inverse): 20%    - Reinfection rate: 15%    - UEBA anomaly correlation: 15%    """    scores = {}    # Active exposures (0-25): scale by logarithm    active = int(metrics.get('ActiveCount', 0))    scores['Active Exposures'] = min(25, int(np.log1p(active) * 3))    # Plaintext percentage (0-25)    plaintext_pct = float(metrics.get('PlaintextPct', 0))    scores['Plaintext Passwords'] = min(25, int(plaintext_pct / 4))    # Remediation rate inverted (0-20): lower remediation = higher risk    remediation_pct = float(metrics.get('RemediationPct', 100))    scores['Remediation Gap'] = int((100 - remediation_pct) / 5)    # Reinfection rate (0-15)    reinfected = int(metrics.get('ReinfectedDevices', 0))    total_devices = max(1, int(metrics.get('TotalDevices', 1)))    reinfection_pct = (reinfected / total_devices) * 100    scores['Reinfection Rate'] = min(15, int(reinfection_pct / 5))    # UEBA correlation (0-15)    ueba_count = int(metrics.get('HighPriorityCorrelated', 0))    scores['UEBA Anomalies'] = min(15, ueba_count * 3)    total = sum(scores.values())    return min(100, total), scoresorg_risk_score, score_breakdown = calculate_org_risk_score(m)# Display risk gaugefig = go.Figure(go.Indicator(    mode="gauge+number+delta",    value=org_risk_score,    title={"text": "Organizational Risk Score"},    delta={"reference": 50, "increasing": {"color": "red"}, "decreasing": {"color": "green"}},    gauge={        "axis": {"range": [0, 100]},        "bar": {"color": "darkblue"},        "steps": [            {"range": [0, 30], "color": "green"},            {"range": [30, 60], "color": "yellow"},            {"range": [60, 80], "color": "orange"},            {"range": [80, 100], "color": "red"}        ],        "threshold": {            "line": {"color": "black", "width": 4},            "thickness": 0.75,            "value": org_risk_score        }    }))fig.update_layout(height=350)fig.show()display(Markdown("### Score Breakdown"))for factor, pts in score_breakdown.items():    bar = "█" * pts + "░" * (25 - pts)    display(Markdown(f"- **{factor}:** {pts} pts `{bar}`"))

In [ ]:
# Risk score trend over time with targetsrisk_trend_query = f"""SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago({LOOKBACK_DAYS}d)| summarize    ActiveExposures = count(),    PlaintextPct = round(100.0 * countif(password_type_s == "plaintext") / count(), 1),    UniqueUsers = dcount(email_s)  by Week = startofweek(TimeGenerated)| order by Week asc"""risk_trend_df = qry_prov.exec_query(risk_trend_query)if risk_trend_df is not None and not risk_trend_df.empty:    # Calculate simplified weekly risk score    risk_trend_df['WeeklyRisk'] = (        risk_trend_df['ActiveExposures'].apply(lambda x: min(25, int(np.log1p(x) * 3))) +        risk_trend_df['PlaintextPct'].apply(lambda x: min(25, int(x / 4)))    )    fig = go.Figure()    fig.add_trace(go.Scatter(        x=risk_trend_df['Week'], y=risk_trend_df['WeeklyRisk'],        mode='lines+markers', name='Risk Score',        line=dict(color='red', width=3)    ))    # Target lines    fig.add_hline(y=30, line_dash="dash", line_color="green",                  annotation_text="Target (<30)")    fig.add_hline(y=60, line_dash="dash", line_color="orange",                  annotation_text="Warning (60)")    fig.add_hline(y=80, line_dash="dash", line_color="red",                  annotation_text="Critical (80)")    fig.update_layout(        title="Organizational Risk Score Trend",        xaxis_title="Week", yaxis_title="Risk Score",        height=400, yaxis_range=[0, 100]    )    fig.show()

## Section 5: Firewall & Network PostureCorrelate SpyCloud threat data with firewall events, DNS anomalies, and VPN risk to assess network security posture.

In [ ]:
# SpyCloud IP overlap with firewall eventsip_overlap_query = f"""let SpyCloudIPs = SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago(30d)| where ip_address_s != ""| distinct ip_address_s;CommonSecurityLog| where TimeGenerated >= ago(30d)| where SourceIP in (SpyCloudIPs) or DestinationIP in (SpyCloudIPs)| summarize    EventCount = count(),    AllowCount = countif(DeviceAction has_any ("allow", "accept", "pass")),    DenyCount = countif(DeviceAction has_any ("deny", "drop", "block", "reject")),    FirstSeen = min(TimeGenerated),    LastSeen = max(TimeGenerated),    UniqueDestPorts = dcount(DestinationPort)  by DeviceVendor, DeviceProduct,     SourceIP, DestinationIP, DeviceAction| order by EventCount desc| take 50"""ip_overlap_df = qry_prov.exec_query(ip_overlap_query)if ip_overlap_df is not None and not ip_overlap_df.empty:    display(Markdown("### SpyCloud IP Overlap with Firewall Events"))    display(Markdown(f"**{len(ip_overlap_df)} IP-level matches found**"))    # Summary by vendor    vendor_summary = ip_overlap_df.groupby('DeviceVendor').agg(        TotalEvents=('EventCount', 'sum'),        AllowedEvents=('AllowCount', 'sum'),        DeniedEvents=('DenyCount', 'sum')    ).reset_index()    fig = px.bar(        vendor_summary, x='DeviceVendor',        y=['AllowedEvents', 'DeniedEvents'],        barmode='stack',        title="Firewall Actions for SpyCloud IPs by Vendor",        color_discrete_map={'AllowedEvents': 'green', 'DeniedEvents': 'red'}    )    fig.show()    display(ip_overlap_df.head(20))else:    print("No firewall events matching SpyCloud IPs found.")

In [ ]:
# DNS anomaly correlationdns_anomaly_query = f"""let InfectedDevices = SpyCloudCompassDevices_CL| where TimeGenerated >= ago(30d)| distinct hostname_s;DnsEvents| where TimeGenerated >= ago(7d)| where Computer in (InfectedDevices)| summarize    QueryCount = count(),    UniqueQueries = dcount(Name),    NXDomainCount = countif(ResultCode == 3),    FirstSeen = min(TimeGenerated),    LastSeen = max(TimeGenerated)  by Computer, SubType| extend AnomalyScore = round(    1.0 * NXDomainCount / max_of(QueryCount, 1) * 100 +    iff(UniqueQueries > 500, 20, 0), 1)| order by AnomalyScore desc"""dns_df = qry_prov.exec_query(dns_anomaly_query)if dns_df is not None and not dns_df.empty:    display(Markdown("### DNS Anomaly Analysis for Infected Hosts"))    fig = px.scatter(        dns_df, x='QueryCount', y='UniqueQueries',        size='AnomalyScore', color='AnomalyScore',        hover_name='Computer',        title="DNS Activity Anomalies (Infected Devices)",        color_continuous_scale='RdYlGn_r',        labels={'QueryCount': 'Total Queries', 'UniqueQueries': 'Unique Domains'}    )    fig.show()    display(dns_df)else:    print("No DNS anomalies detected.")

In [ ]:
# VPN risk assessmentvpn_query = f"""let ExposedUsers = SpyCloudBreachWatchlist_CL| where TimeGenerated >= ago(30d)| distinct email_s;SigninLogs| where TimeGenerated >= ago(30d)| where ClientAppUsed has_any ("VPN", "Remote Desktop", "SSTP", "IKEv2")    or AppDisplayName has_any ("VPN", "Remote Access", "GlobalProtect",                                "Cisco AnyConnect", "Pulse Secure")| where UserPrincipalName in (ExposedUsers)| summarize    VPNSignins = count(),    UniqueIPs = dcount(IPAddress),    Locations = make_set(strcat(LocationDetails.city, ", ",                                LocationDetails.countryOrRegion)),    RiskyCount = countif(RiskLevelDuringSignIn in ("high", "medium")),    LastVPNAccess = max(TimeGenerated)  by UserPrincipalName, AppDisplayName| order by VPNSignins desc"""vpn_df = qry_prov.exec_query(vpn_query)if vpn_df is not None and not vpn_df.empty:    display(Markdown("### VPN Risk Assessment"))    display(Markdown("Exposed users with VPN/Remote access activity:"))    fig = px.bar(        vpn_df, x='UserPrincipalName', y='VPNSignins',        color='RiskyCount', color_continuous_scale='Reds',        title="VPN Usage by Exposed Users",        hover_data=['AppDisplayName', 'UniqueIPs']    )    fig.update_layout(xaxis_tickangle=-45, height=450)    fig.show()    display(vpn_df)else:    print("No VPN risk indicators found for exposed users.")

## Section 6: Executive ReportAuto-generated executive summary with key metrics, KPIs, and actionable recommendations. Designed for PDF export.

In [ ]:
# Auto-generated executive summaryreport_date = datetime.utcnow().strftime("%B %d, %Y")display(HTML(f"""<div style="border: 2px solid #333; padding: 30px; font-family: Arial, sans-serif;            max-width: 800px; margin: auto; background: white;">    <h1 style="text-align: center; color: #1a1a2e;">SpyCloud Threat Landscape Report</h1>    <h3 style="text-align: center; color: #666;">Generated: {report_date}</h3>    <h3 style="text-align: center; color: #666;">Analysis Period: {LOOKBACK_DAYS} Days</h3>    <hr>    <h2 style="color: #1a1a2e;">Executive Summary</h2>    <p>This report provides an organizational overview of credential exposure risks    identified through SpyCloud's breach and malware intelligence platform, correlated    with Microsoft Sentinel security telemetry.</p>    <h3 style="color: #e94560;">Risk Score: {org_risk_score}/100</h3>    <h2 style="color: #1a1a2e;">Key Findings</h2>    <ul>        <li><strong>Active Exposures:</strong> {int(m.get('ActiveCount', 0)):,} records            affecting {int(m.get('ActiveUsers', 0)):,} unique users</li>        <li><strong>Plaintext Password Exposure:</strong> {float(m.get('PlaintextPct', 0)):.1f}%            of exposed credentials are in plaintext</li>        <li><strong>Remediation Rate:</strong> {float(m.get('RemediationPct', 0)):.1f}%            of affected users have been remediated</li>        <li><strong>Reinfected Devices:</strong> {int(m.get('ReinfectedDevices', 0))}            out of {int(m.get('TotalDevices', 0))} infected devices</li>        <li><strong>UEBA Correlation:</strong> {int(m.get('HighPriorityCorrelated', 0))}            high-priority behavioral anomalies correlated with exposed users</li>    </ul></div>"""))

In [ ]:
# Key metrics and KPIsdisplay(HTML("""<div style="border: 2px solid #333; padding: 30px; font-family: Arial, sans-serif;            max-width: 800px; margin: auto; background: white;">    <h2 style="color: #1a1a2e;">Key Performance Indicators</h2></div>"""))# KPI cardskpis = {    "Mean Time to Remediation": "Measures average time between SpyCloud alert and password reset",    "Exposure Coverage Rate": "Percentage of org email domains monitored by SpyCloud",    "Reinfection Rate": "Percentage of previously remediated devices re-infected",    "Credential Exposure Density": "Average exposures per user across the organization",    "Plaintext Reduction Rate": "Month-over-month reduction in plaintext exposures"}# Calculate KPIs from available dataif total_stats is not None and not total_stats.empty:    stats = total_stats.iloc[0]    total_exp = int(stats.get('TotalExposures', 0))    unique_users = max(1, int(stats.get('UniqueUsers', 1)))    density = round(total_exp / unique_users, 1)    kpi_data = pd.DataFrame({        'KPI': ['Credential Exposure Density', 'Unique Users Affected',                'Total Breaches Tracked', 'Plaintext Exposure %',                'Remediation Coverage %'],        'Value': [density, unique_users, int(stats.get('UniqueBreaches', 0)),                  float(m.get('PlaintextPct', 0)),                  float(m.get('RemediationPct', 0))],        'Target': [1.0, 0, 0, 0, 100],        'Status': ['On Track' if density < 2 else 'At Risk',                   'Monitoring', 'Monitoring',                   'On Track' if float(m.get('PlaintextPct', 0)) < 10 else 'At Risk',                   'On Track' if float(m.get('RemediationPct', 0)) > 80 else 'At Risk']    })    display(kpi_data.style.apply(        lambda x: ['background-color: #d4edda' if v == 'On Track'                   else 'background-color: #f8d7da' if v == 'At Risk'                   else '' for v in x],        subset=['Status']    ))

In [ ]:
# Top recommendationsdisplay(HTML("""<div style="border: 2px solid #333; padding: 30px; font-family: Arial, sans-serif;            max-width: 800px; margin: auto; background: white;">    <h2 style="color: #1a1a2e;">Recommendations</h2></div>"""))recommendations = []# Generate recommendations based on dataplaintext_pct = float(m.get('PlaintextPct', 0))remediation_pct = float(m.get('RemediationPct', 0))reinfected = int(m.get('ReinfectedDevices', 0))ueba_correlated = int(m.get('HighPriorityCorrelated', 0))if plaintext_pct > 20:    recommendations.append({        'Priority': 'CRITICAL',        'Area': 'Password Policy',        'Recommendation': f'{plaintext_pct:.0f}% of credentials exposed in plaintext. '                          'Enforce complex password policies and accelerate migration to '                          'passwordless authentication (FIDO2, Windows Hello).',        'Impact': 'High'    })if remediation_pct < 80:    recommendations.append({        'Priority': 'CRITICAL',        'Area': 'Remediation',        'Recommendation': f'Only {remediation_pct:.0f}% remediation rate. '                          'Automate password reset and MFA re-registration via '                          'Conditional Access policies triggered by SpyCloud alerts.',        'Impact': 'High'    })if reinfected > 0:    recommendations.append({        'Priority': 'HIGH',        'Area': 'Device Security',        'Recommendation': f'{reinfected} devices show reinfection. '                          'Implement aggressive endpoint hardening, review '                          'EDR policies, and consider device replacement for '                          'persistently infected machines.',        'Impact': 'High'    })if ueba_correlated > 0:    recommendations.append({        'Priority': 'HIGH',        'Area': 'Identity Protection',        'Recommendation': f'{ueba_correlated} behavioral anomalies correlated with '                          'exposed users. Enable real-time risk-based Conditional Access '                          'and investigate high-priority alerts immediately.',        'Impact': 'Medium'    })recommendations.extend([    {        'Priority': 'MEDIUM',        'Area': 'Monitoring',        'Recommendation': 'Expand SpyCloud monitoring to cover all organizational '                          'domains and third-party service accounts.',        'Impact': 'Medium'    },    {        'Priority': 'MEDIUM',        'Area': 'Training',        'Recommendation': 'Deploy targeted security awareness training for users '                          'with repeated credential exposures.',        'Impact': 'Medium'    },    {        'Priority': 'LOW',        'Area': 'Compliance',        'Recommendation': 'Review and update incident response playbooks to include '                          'SpyCloud-triggered credential exposure scenarios.',        'Impact': 'Low'    }])rec_df = pd.DataFrame(recommendations)display(rec_df.style.apply(    lambda x: ['background-color: #f8d7da' if v == 'CRITICAL'               else 'background-color: #fff3cd' if v == 'HIGH'               else 'background-color: #d4edda' if v == 'LOW'               else '' for v in x],    subset=['Priority']))

In [ ]:
# Exportable report datareport_export = {    "report_date": report_date,    "analysis_period_days": LOOKBACK_DAYS,    "organization_risk_score": org_risk_score,    "risk_breakdown": score_breakdown,    "key_metrics": {        "active_exposures": int(m.get('ActiveCount', 0)),        "unique_affected_users": int(m.get('ActiveUsers', 0)),        "plaintext_password_pct": float(m.get('PlaintextPct', 0)),        "remediation_rate_pct": float(m.get('RemediationPct', 0)),        "reinfected_devices": int(m.get('ReinfectedDevices', 0)),        "ueba_correlated_anomalies": int(m.get('HighPriorityCorrelated', 0)),    },    "recommendations": recommendations,    "generated_utc": datetime.utcnow().isoformat(),}import jsonreport_path = f"spycloud_threat_landscape_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"with open(report_path, 'w') as f:    json.dump(report_export, f, indent=2, default=str)display(Markdown(f"### Report exported to: `{report_path}`"))display(Markdown("**Tip:** Use `jupyter nbconvert --to pdf` to generate a PDF version of this notebook."))print("\n" + "=" * 60)print("SpyCloud Threat Landscape Analysis Complete")print("=" * 60)

---**End of SpyCloud Threat Landscape Analysis Notebook**For questions or issues, contact the SOC engineering team.